In [1]:
import numpy as np
import pandas as pd
import xarray as xr

from pathlib import Path

In [2]:
regions = np.arange(1,20,1)
path = '/scratch/k10/wy2165/PyGEM/'
outpath = '/g/data/rd53/wy2165/disequilibrium/pygem_oggm/'
datapath = '/g/data/rd53/wy2165/disequilibrium/data/'

result = pd.read_csv(datapath + 'temp_ch_ipcc_ar6_isimip3b.csv', index_col=0)
result['n_glacier'] = 215547
result['all_steady'] = np.nan
result['all_mean'] = np.nan
result['MCMC_calibration_failed'] = 900
result['run_simulation_2000_2025'] = np.nan
result['run_simulations'] = np.nan
result['AAR_steady'] = np.nan
result['AAR_mean'] = np.nan

In [3]:
for i in range(81):
    gcm = result['gcm'].values[i]
    period_scenario = result['period_scenario'].values[i]
    ssp = period_scenario[10:]

    if ssp == 'hist':
        ssp = 'ssp126'

    all_steady = 0
    all_mean = 0
    run_simulation_2000_2025 = 0
    run_simulations = 0
    AAR_steady = 0
    AAR_mean = 0
    for region in regions:
        # all_steady
        if i == 0:
            filepath = path + f'oggm_gdirs/OGGM/{region:02d}/glacier_AAR_steady_PyGEM_{region:02d}_{gcm.upper()}.csv'
        else:
            filepath = path + f'oggm_gdirs/OGGM/{region:02d}/glacier_AAR_steady_PyGEM_{region:02d}_{gcm.upper()}_{ssp}.csv'
        
        data = pd.read_csv(filepath)
        all_steady = all_steady + np.sum(np.isnan(data['AAR_steady'])).item()

        # all_mean
        if i == 0:
            filepath = path + f'/oggm_gdirs/None/{region:02d}/glacier_disequilibrium_PyGEM_{region:02d}_{gcm.upper()}.csv'
        else:
            filepath = path + f'/oggm_gdirs/None/{region:02d}/glacier_disequilibrium_PyGEM_{region:02d}_{gcm.upper()}_{ssp}.csv'
        
        data = pd.read_csv(filepath)
        column_name = gcm + '_' + period_scenario + '_AAR_mean'
        all_mean = all_mean + np.sum(np.isnan(data[column_name])).item()

        # run_simulation_2000_2025
        if i == 0:
            folder = path + f'Output/simulations_oggm/failed/{region:02d}/{gcm.upper()}/'
        else:
            folder = path + f'Output/simulations_oggm/failed/{region:02d}/{gcm.upper()}/{ssp}/'

        folder = Path(folder)
        if folder.exists():
            run_simulation_2000_2025 = run_simulation_2000_2025 + len([p for p in folder.iterdir() if p.is_file()])
        else:
            run_simulation_2000_2025 = run_simulation_2000_2025 + 0

        # run_simulations
        if i == 0:
            folder = path + f'Output/simulations_none/failed/{region:02d}/{gcm.upper()}/'
        else:
            folder = path + f'Output/simulations_none/failed/{region:02d}/{gcm.upper()}/{ssp}/'

        folder = Path(folder)
        if folder.exists():
            run_simulations = run_simulations + len([p for p in folder.iterdir() if p.is_file()])
        else:
            run_simulations = run_simulations + 0

    run_simulation_2000_2025 = run_simulation_2000_2025 - 900
    run_simulations = run_simulations - 900

    AAR_steady = all_steady - run_simulation_2000_2025 - 900
    AAR_mean   = all_mean - run_simulations - 900

    result.loc[i, 'all_steady'] = all_steady
    result.loc[i, 'all_mean'] = all_mean
    result.loc[i, 'run_simulation_2000_2025'] = run_simulation_2000_2025
    result.loc[i, 'run_simulations'] = run_simulations
    result.loc[i, 'AAR_steady'] = AAR_steady
    result.loc[i, 'AAR_mean'] = AAR_mean

In [4]:
result.to_csv(outpath + f'PyGEM_faileds.csv')